In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# pip install gensim
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(400)

In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
df = pd.read_csv("/content/2020_Sep6_154k_hydrated_tweets.csv")

In [ ]:
df.drop(columns=['id_str',"in_reply_to_screen_name","is_retweet"], inplace=True)

In [ ]:
# Function to Clean the Tweet.

import re
def clean_tweet(tweet):
    return ' '.join(re.sub('(\\\\n)|(b\"[^0-9A-Za-z A-Za-z0-9 \t]+)|(b\'[^0-9A-Za-z]+)|(b\"[A-Za-z0-9]+)|(b\'[A-Za-z0-9]+)|(b\'#[A-Za-z0-9]+)|(b\'@[A-Za-z0-9]+)|(\\\\x[A-Za-z0-9]+)|(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|([RT])', ' ', str(tweet).lower()).split())


In [ ]:
# Call function to get Clean tweets
df["CleanTweet"] = df['text'].apply(lambda x : clean_tweet(x))
df.tail()

,favorite_count,source,text,created_at,retweet_count,CleanTweet
153749,1,Twitter for iPad,b'@theasscat I even get fooled sometimes. It\x...,Mon Sep 07 04:03:13 +0000 2020,0,theasscat i even get fooled sometimes it becom...
153750,4,COVID-19MX,b'#ACTUALIZACI\xc3\x93N GLOBAL \xf0\x9f\x8c\x8...,Mon Sep 07 04:03:15 +0000 2020,2,actualizaci global top casos 6 460 250 4 202 5...
153751,0,erased16993361,b'#Covid19 bot \n#COVID19 last 5 days stats #U...,Mon Sep 07 04:03:06 +0000 2020,0,covid19 bot covid19 last 5 days stats uttarakh...
153752,0,SocialFlow,"b""Coronavirus: What's open, what's closed in t...",Mon Sep 07 04:03:15 +0000 2020,0,what s open what s closed in the mid hudson va...
153753,0,Twitter for iPhone,b'@KellyannePolls is a very disgusting person ...,Mon Sep 07 04:03:13 +0000 2020,0,kellyannepolls is a very disgusting person


In [ ]:
df.drop(df.index[df.CleanTweet.eq("")], inplace=True)

In [ ]:
df.head()

,favorite_count,source,text,created_at,retweet_count,CleanTweet
0,1,Twitter for Android,"b""Don't believe it. https://t.co/VFfhPArn1a""",Sat Sep 03 04:09:04 +0000 2022,0,t believe it
1,0,cmssocialservice,b'Surat records 11 new Covid-19 cases https://...,Sat Sep 03 04:06:03 +0000 2022,0,records 11 new covid 19 cases
2,38,TweetDeck,b'Hennepin County will no longer require COVID...,Sat Sep 03 04:09:00 +0000 2022,6,county will no longer require covid 19 vaccine...
3,2,Twitter Web App,"b'As of Sept 1, the capital has given over 63....",Sat Sep 03 04:08:52 +0000 2022,0,of sept 1 the capital has given over 63 2 mill...
4,1,Twitter for Android,b'@KushThrough @BlazedRTs @rtsmallstreams @Rts...,Sat Sep 03 04:15:46 +0000 2022,3,kushthrough ww nowplaying x lilroyce her ev


In [ ]:
stemmer = SnowballStemmer(language='english')
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    result=[]
    for token in gensim.utils.simple_preprocess(text) :
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
            
    return result

In [ ]:
complete_text = ' '.join(df["CleanTweet"])

In [ ]:
data = df.CleanTweet.values.tolist()

In [ ]:
pprint(data[:2])

Pretty printing has been turned OFF


In [ ]:
complete_text


't believe it records 11 new covid 19 cases county will no longer require covid 19 vaccines for its employees of sept 1 the capital has given over 63 2 million doses of covid19 vaccines to over 23 6 million people kushthrough ww nowplaying x lilroyce her ev can t be good for you themixmedic nowplaying x lilroyce her evil spine feat makrazy total of 174 new cases of covid 19 were reported during the last 24 hours across the state odisha odishanews appointed by the israeli moh to investigate covid 19 vaccine side effects warned the ministry it could b covid 19 closed our campuses down the idea that every student has a computer a word processing program sta thetruthsucks12 why get poked in 2021 when covid 19 has a 98 survival rate got a bridge in brooklyn for sale yo federal government is fucking trash dubai reins in hospitality as covid 19 cases rise is hilarious you can make this up the camp counselor who tried to overturn a decisive democratic election pminmangaluru pm modi mentioned t

In [ ]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
processed_docs = preprocess(complete_text)

In [ ]:
dictionary = gensim.corpora.Dictionary([processed_docs])

In [ ]:
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 aaaaaaaaa
1 aajtak
2 aapi
3 aapl
4 aarogya
5 aaron
6 aarondodd
7 aaronla
8 aaronparna
9 aarospeir
10 aayeff


In [ ]:
#dictionary.filter_extremes(no_below=20, no_above=0.1, keep_n= 1000000)
#dictionary.filter_extremes(no_above=0.70)

In [ ]:
dictionary

In [ ]:
#bow_corpus = dictionary.doc2bow(processed_docs)

bow_corpus = [dictionary.doc2bow(processed_docs),]
#bow_corpus = [dictionary.doc2bow(text) for text in processed_docs]
print(bow_corpus[:1])

[[]]


In [ ]:
#LDA
lda_model = gensim.models.ldamodel.LdaModel(corpus=bow_corpus, 
                                            num_topics = 20, 
                                            id2word = dictionary,
                                            random_state=100,
                                            update_every=1,
                                            chunksize=100,
                                            alpha='auto',
                                            per_word_topics=True,
                                            passes = 50)

In [ ]:
for idx, topic in lda_model.print_topics(-1):
    print("Topic: {} \nWords: {}".format(idx, topic ))
    print("\n")

In [ ]:
import re
import numpy as np
import pandas as  pd
from pprint import pprint# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
#from gensim.models import CoherenceModel# spaCy for preprocessing
import spacy# Plotting tools
#import pyLDAvis
#import pyLDAvis.gensim
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
!pip3 install spacy

!python3 -m spacy download en #Language model

#pip3 install gensim # For topic modeling

#pip3 install pyLDAvis # For visualizing topic models

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
2022-10-06 08:42:33.861063: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 12.8 MB 7.4 MB/s 
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
# NLTK Stop words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

In [ ]:
def sent_to_words(sentences):
  for sentence in sentences:
    yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))            #deacc=True removes punctuations

In [ ]:
data_words = list(sent_to_words(data))
print(data_words[:1])

[['and', 'we', 'should', 'panic', 'and', 'not', 'send', 'the', 'kids', 'to', 'school', 'cbc', 'presstitutes', 'all', 'imho']]


In [ ]:
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)
print(trigram_mod[bigram_mod[data_words[0]]])


/usr/local/lib/python3.7/dist-packages/gensim/models/phrases.py:598: UserWarning: For a faster implementation, use the gensim.models.phrases.Phraser class
  warnings.warn("For a faster implementation, use the gensim.models.phrases.Phraser class")


['and', 'we', 'should', 'panic', 'and', 'not', 'send', 'the', 'kids', 'to', 'school', 'cbc', 'presstitutes', 'all', 'imho']


In [ ]:
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [ ]:
data_words_nostops = remove_stopwords(data_words)

data_words_bigrams = make_bigrams(data_words_nostops)

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [ ]:
print(data_lemmatized[:5])

[['panic', 'send', 'kid', 'school', 'cbc', 'presstitute', 'imho'], ['say', 'last', 'forever', 'maybe', 'go', 'away'], ['explain'], ['ericcolumbus', 'article', 'say', 'positivity'], ['thedailybeast', 'surprise', 'absurd', 'think', 'expect', 'university', 'student', 'stay']]


In [ ]:
id2word = corpora.Dictionary(data_lemmatized)  
texts = data_lemmatized  
corpus = [id2word.doc2bow(text) for text in texts]  
print(corpus[:1])

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1)]]


In [ ]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20, 
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=50,
                                           alpha='auto',
                                           per_word_topics=True)

In [ ]:
# Print the keyword of topics
from pprint import pprint
pprint(lda_model.print_topics())

[(0,
  '0.142*"get" + 0.035*"student" + 0.023*"lie" + 0.022*"talk" + 0.021*"return" '
  '+ 0.021*"public" + 0.017*"pay" + 0.016*"rule" + 0.015*"part" + '
  '0.015*"hold"'),
 (1,
  '0.074*"news" + 0.063*"year" + 0.053*"think" + 0.046*"way" + 0.027*"turn" + '
  '0.026*"result" + 0.025*"expect" + 0.022*"order" + 0.018*"fuck" + '
  '0.018*"stay"'),
 (2,
  '0.487*"covid" + 0.052*"test" + 0.043*"day" + 0.034*"positive" + '
  '0.034*"state" + 0.030*"update" + 0.017*"rise" + 0.016*"testing" + '
  '0.015*"high" + 0.013*"right"'),
 (3,
  '0.148*"death" + 0.069*"amp" + 0.055*"number" + 0.042*"late" + '
  '0.029*"reopen" + 0.028*"infection" + 0.022*"run" + 0.020*"travel" + '
  '0.020*"already" + 0.017*"seem"'),
 (4,
  '0.316*"case" + 0.156*"coronavirus" + 0.131*"new" + 0.058*"report" + '
  '0.029*"first" + 0.017*"hospital" + 0.016*"recovery" + 0.012*"quarantine" + '
  '0.010*"away" + 0.009*"supply"'),
 (5,
  '0.065*"mask" + 0.062*"make" + 0.046*"lockdown" + 0.033*"tell" + '
  '0.027*"strategy" + 0

In [ ]:
doc_lda = lda_model[corpus]

In [ ]:
# Compute Perplexity
print('\nPerplexity: ', lda_model.log_perplexity(corpus))  
# a measure of how good the model is. lower the better.


Perplexity:  -9.5473275827816


In [ ]:
# Compute Coherence Score
from gensim.models import CoherenceModel# spaCy for preprocessing
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Coherence Score:  0.32399096642698055


In [ ]:
!pip3 install pyLDAvis==2.1.2

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.6 MB 8.2 MB/s 
  Created wheel for pyLDAvis: filename=pyLDAvis-2.1.2-py2.py3-none-any.whl size=97738 sha256=b2f5ee9b2ab4de43e7701464186c6d653535194e70c0b9291c57afd8007bd629
  Stored in directory: /root/.cache/pip/wheels/3b/fb/41/e32e5312da9f440d34c4eff0d2207b46dc9332a7b931ef1e89
Successfully built pyLDAvis


In [ ]:
for index, topic in lda_model.show_topics(formatted=False, num_words= 50):
    print('Topic: {} \nWords: {}'.format(index, [w[0] for w in topic]))

Topic: 8 
Words: ['confirm', 'last', 'week', 'trend_sep', 'restriction', 'resume', 'post', 'postpone', 'research', 'days_stat', 'move', 'soar', 'donaldjtrumpjr', 'funny', 'pressure', 'overall', 'past', 'position', 'strain', 'church', 'lag', 'johns_hopkin', 'currently', 'film', 'obsess', 'lost_control', 'straw', 'fix', 'ad', 'experiment', 'south_wale', 'domestic', 'customer', 'size', 'daddy', 'hrs', 'politicize', 'temporarily', 'rooker_detail', 'verify', 'grant', 'minute', 'forward', 'yo', 'encourage', 'user', 'playoff', 'updates_globally', 'fox', 'zoom']
Topic: 10 
Words: ['time', 'record', 'realdonaldtrump', 'patient', 'kill', 'link', 'community', 'situation', 'push', 'sure', 'trend', 'play', 'page', 'citizen', 'gossip', 'last_hour', 'fresh', 'politic', 'forever', 'combat', 'yesterday', 'form', 'bank', 'partner', 'wave', 'golf', 'misinformation', 'requirement', 'theory', 'maine_wedding', 'review', 'typical', 'scott_morrison', 'plague', 'twitter', 'face_bleak', 'takes_toll', 'describe'

In [ ]:
for idx, topic in lda_model.show_topics(formatted=False, num_topics=20, num_words= 100):
    print('Topic: {} \nWords: {}'.format(idx, '|'.join([w[0] for w in topic])))


Topic: 0 
Words: get|student|lie|talk|return|public|pay|rule|part|hold|warn|guy|college|app|social|reduce|labor|forget|food|sick|large|act|write|eat|finally|rest|suspend|contact|experience|emergency|opportunity|explain|front|smartnew|fan|football|search|nyu_student|energy|design|ban|centre|wit|endless|critical|dream|elect|bill|indeed|kick|violate|trip|gather|drink|hotel|enter|weather|weak|operate|restaurant|maintain|flight|crazy|hotspot|task|lawsuit|property|opt|rent|comfort|trumpvirus|celebration|usopen|joy|adapt|joebiden|famine|democracy|potus|sort|detect|endure|university_dismisse|enforce|shelter|necessary|dis|stay_safe|shoot|housing|op|smh|insurance|signed_petition|tuition_fee|stock|value|respect|phone|degree
Topic: 1 
Words: news|year|think|way|turn|result|expect|order|fuck|stay|believe|low|university|feel|american|event|figure|especially|serious|quite|holiday|address|exam|thousand|whole|tax|indian|governor|responsible|fool|management|appear|passenger|surprise|covidiot|conversatio

In [ ]:
from gensim.parsing.preprocessing import preprocess_string, strip_punctuation, strip_numeric

lda_topics = lda_model.show_topics(num_topics=20, num_words=50)

topics = []
filters = [lambda x: x.lower(), strip_punctuation, strip_numeric]

for topic in lda_topics:
    print(topic)
    topics.append(preprocess_string(topic[1], filters))

print(topics)

(0, '0.142*"get" + 0.035*"student" + 0.023*"lie" + 0.022*"talk" + 0.021*"return" + 0.021*"public" + 0.017*"pay" + 0.016*"rule" + 0.015*"part" + 0.015*"hold" + 0.014*"warn" + 0.013*"guy" + 0.012*"college" + 0.011*"app" + 0.011*"social" + 0.011*"reduce" + 0.011*"labor" + 0.010*"forget" + 0.010*"food" + 0.010*"sick" + 0.009*"large" + 0.008*"act" + 0.008*"write" + 0.008*"eat" + 0.007*"finally" + 0.007*"rest" + 0.007*"suspend" + 0.007*"contact" + 0.007*"experience" + 0.006*"emergency" + 0.006*"opportunity" + 0.006*"explain" + 0.005*"front" + 0.005*"smartnew" + 0.005*"fan" + 0.005*"football" + 0.005*"search" + 0.005*"nyu_student" + 0.004*"energy" + 0.004*"design" + 0.004*"ban" + 0.004*"centre" + 0.004*"wit" + 0.004*"endless" + 0.004*"critical" + 0.004*"dream" + 0.004*"elect" + 0.004*"bill" + 0.004*"indeed" + 0.004*"kick"')
(1, '0.074*"news" + 0.063*"year" + 0.053*"think" + 0.046*"way" + 0.027*"turn" + 0.026*"result" + 0.025*"expect" + 0.022*"order" + 0.018*"fuck" + 0.018*"stay" + 0.018*"beli

In [ ]:
# Visualize the topics
import pyLDAvis.gensim
#import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

/usr/local/lib/python3.7/dist-packages/past/types/oldstr.py:5: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  from collections import Iterable
/usr/local/lib/python3.7/dist-packages/pyLDAvis/_prepare.py:232: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  head(R).drop('saliency', 1)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.419772  0.099648       1        1  10.394471
11    -0.073429 -0.415107       2        1   7.378781
4     -0.040852  0.027444       3        1   6.528569
18    -0.018938 -0.014618       4        1   6.092853
12     0.022138  0.019429       5        1   4.944869
3      0.023005  0.016846       6        1   4.914258
7      0.008746 -0.008724       7        1   4.900220
19     0.012027  0.010083       8        1   4.853266
17     0.010580  0.013499       9        1   4.795449
0      0.036529  0.023238      10        1   4.686930
9      0.039458  0.012830      11        1   4.683331
15     0.041419  0.021055      12        1   4.477242
13     0.038851  0.020857      13        1   4.438821
1      0.037674  0.024291      14        1   4.385196
14     0.033280  0.021632      15        1   4.367525
5      0.049171  0.021977      16        1   4.202023
16     0.042730  0.021997      17        1   3.987820
6      0.037225  0.023594      18        1   3.966856
10     0.058528  0.026904      19        1   3.421439
8      0.061629  0.033125      20        1   2.580081, topic_info=               Term          Freq         Total Category  logprob  loglift
26            covid  45437.000000  45437.000000  Default  30.0000  30.0000
25             case  18518.000000  18518.000000  Default  29.0000  29.0000
73      coronavirus   9130.000000   9130.000000  Default  28.0000  28.0000
74              new   7666.000000   7666.000000  Default  27.0000  27.0000
57           people   6492.000000   6492.000000  Default  26.0000  26.0000
...             ...           ...           ...      ...      ...      ...
8711           size     70.826796     71.747709  Topic20  -5.7908   3.6444
21664         straw     96.595508    100.830849  Topic20  -5.4805   3.6144
21435  lost_control    106.977541    114.328729  Topic20  -5.3784   3.5909
38        trend_sep    885.050472   2155.385873  Topic20  -3.2654   2.7673
264       days_stat    391.304050   1176.930060  Topic20  -4.0815   2.5562

[635 rows x 6 columns], token_table=      Topic      Freq       Term
term                            
846       9  0.997676       able
3324      6  0.998108     access
92        4  0.997412     accord
2344     10  0.998477        act
613      18  0.996670     action
...     ...       ...        ...
317      14  0.999700       year
2457     19  0.993929  yesterday
3462     11  0.998031        yet
2956      3  0.992734  york_time
586       2  0.998514      young

[644 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 12, 5, 19, 13, 4, 8, 20, 18, 1, 10, 16, 14, 2, 15, 6, 17, 7, 11, 9])

In [ ]:
pyLDAvis.save_html(vis, '2020_lda.html')

In [ ]:
!unzip "/content/mallet-2.0.8.zip"

In [ ]:
mallet_path = '/content/mallet-2.0.8/bin/mallet' # update this path
ldamallet = gensim.models.wrappers.LdaMallet(mallet_path, 
                                             corpus=corpus, 
                                             num_topics=20,
                                             id2word=id2word,
                                             iterations=1000,
                                             optimize_interval=10)

/usr/local/lib/python3.7/dist-packages/smart_open/smart_open_lib.py:494: DeprecationWarning: This function is deprecated.  See https://github.com/RaRe-Technologies/smart_open/blob/develop/MIGRATING_FROM_OLDER_VERSIONS.rst for more information
  warnings.warn(message, category=DeprecationWarning)
/usr/local/lib/python3.7/dist-packages/smart_open/smart_open_lib.py:494: DeprecationWarning: This function is deprecated.  See https://github.com/RaRe-Technologies/smart_open/blob/develop/MIGRATING_FROM_OLDER_VERSIONS.rst for more information
  warnings.warn(message, category=DeprecationWarning)


In [ ]:
pprint(ldamallet.show_topics(num_topics=20,num_words=100,formatted=False))

# Compute Coherence Score
coherence_model_ldamallet = CoherenceModel(model=ldamallet, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_ldamallet = coherence_model_ldamallet.get_coherence()

print('\n Coherence Score: ', round(coherence_ldamallet, 2))

[(0,
  [('amp', 0.10911680911680911),
   ('job', 0.02773892773892774),
   ('home', 0.02584822584822585),
   ('change', 0.024294224294224294),
   ('lie', 0.022584822584822584),
   ('stay', 0.021756021756021756),
   ('love', 0.016524216524216526),
   ('lose', 0.016083916083916083),
   ('time', 0.015591815591815592),
   ('due', 0.013752913752913752),
   ('nation', 0.0114996114996115),
   ('hear', 0.010463610463610464),
   ('big', 0.010282310282310282),
   ('food', 0.009867909867909868),
   ('family', 0.009634809634809635),
   ('air', 0.009194509194509195),
   ('safe', 0.00909090909090909),
   ('season', 0.009013209013209013),
   ('football', 0.008650608650608651),
   ('lack', 0.008547008547008548),
   ('truth', 0.008495208495208495),
   ('order', 0.008262108262108263),
   ('law', 0.00821030821030821),
   ('listen', 0.0077182077182077185),
   ('speak', 0.007381507381507382),
   ('buy', 0.007070707070707071),
   ('thing', 0.006915306915306916),
   ('expose', 0.006785806785806786),
   ('incl

In [ ]:
mallet_lda_model = gensim.models.wrappers.ldamallet.malletmodel2ldamodel(ldamallet)

In [ ]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(mallet_lda_model, corpus, id2word,sort_topics=False)

In [ ]:
print(vis.topic_order)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [ ]:
pyLDAvis.save_html(vis, '2020_lda_mallet.html')

In [ ]:
vis

PreparedData(topic_coordinates=              x         y  topics  cluster      Freq
topic                                               
0     -0.000264  0.000099       1        1  5.012854
1      0.000291 -0.000436       2        1  5.007553
2     -0.000028 -0.000379       3        1  4.994728
3      0.000461  0.000354       4        1  4.992379
4      0.000268 -0.000135       5        1  5.008141
5     -0.000023  0.000329       6        1  5.005779
6     -0.000088  0.000582       7        1  4.989459
7     -0.000171 -0.000448       8        1  4.968436
8     -0.000479  0.000234       9        1  4.994311
9     -0.000542 -0.000165      10        1  4.981681
10     0.000095  0.000053      11        1  5.024642
11    -0.000102 -0.000461      12        1  5.086722
12    -0.000945  0.000127      13        1  5.020679
13     0.000296 -0.000391      14        1  4.978748
14    -0.000083  0.000150      15        1  4.977159
15     0.000185  0.000245      16        1  5.022591
16     0.000593  0.000832      17        1  4.965526
17    -0.000048  0.000099      18        1  4.985264
18     0.000588 -0.000541      19        1  4.980900
19    -0.000004 -0.000150      20        1  5.002448, topic_info=                    Term       Freq      Total Category  logprob  loglift
10755            doubter  20.000000  20.000000  Default  30.0000  30.0000
18911             shroud  20.000000  20.000000  Default  29.0000  29.0000
37996                vmt  21.000000  21.000000  Default  28.0000  28.0000
14655      abigailmarone  20.000000  20.000000  Default  27.0000  27.0000
4467                 goi  20.000000  20.000000  Default  26.0000  26.0000
...                  ...        ...        ...      ...      ...      ...
11763       superstition   1.375044  20.991722  Topic20 -10.3946   0.2696
42382    contentcreation   1.375518  21.196554  Topic20 -10.3943   0.2602
10992           integral   1.368845  20.850532  Topic20 -10.3992   0.2718
20554  overseas_filipino   1.370942  21.420728  Topic20 -10.3976   0.2464
6891                copd   1.370305  21.374703  Topic20 -10.3981   0.2481

[834 rows x 6 columns], token_table=       Topic      Freq           Term
term                                 
35983      1  0.049765         abantu
35983      2  0.049765         abantu
35983      3  0.049765         abantu
35983      4  0.049765         abantu
35983      5  0.049765         abantu
...      ...       ...            ...
20636     16  0.047263  zziwafredrick
20636     17  0.047263  zziwafredrick
20636     18  0.047263  zziwafredrick
20636     19  0.047263  zziwafredrick
20636     20  0.047263  zziwafredrick

[16380 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])